## Evaluation

##  1. Evaluating Summaries (LLM-as-Judge) 

Since source is Wiki text, we have clean ground truth. Use LLM as judge given the summary and source text:
#### What Your LLM Judge Covers
* Faithfulness — does the summary only use what's in the source text?
* Coverage — given the source text available, did the summary capture the important themes? (accounts for thin wikis fairly)
* Tagline accuracy — is the tagline grounded in the source, specific to this metro, not generic?

In [ ]:
# standard library imports 
import os
import sys
import re
import json
import time
import random
import logging

# third-party imports
import boto3
import pandas as pd

# path setup
sys.path.append(os.path.abspath(".."))

# project imports
from src import semantic_search
from src.rag_explanation import generate_explanation
from src.recommender import recommend_cities, add_text_to_cbsa

In [21]:
# ── CONFIG ────────────────────────────────────────────────────────────────────

INPUT_FILE       = "../data/processed/cbsa_wiki_wikivoyage_summaries_df.csv"
OUTPUT_CSV       = "../data/evaluation/summary_eval_results.csv"

SAMPLE_SIZE      = 100    # out of 900+; stratified across small/medium/large
RANDOM_SEED      = 42
SLEEP_BETWEEN    = 0.3   # seconds between API calls — avoids rate limits

# Column names in your CSV — adjust if different
COL_CBSA         = "cbsa_name"
COL_SOURCE       = "city_wiki_wikivoyage_text"
COL_SUMMARY      = "summary"
COL_TAGLINE      = "tagline"

In [22]:
# ── LLM-AS-JUDGE ─────────────────────────────────────────────────────────────

JUDGE_PROMPT = """You are evaluating an AI-generated city summary for a relocation tool.
The summary was generated ONLY from the source wiki text below.

SOURCE TEXT (ground truth):
{source_text}

GENERATED TAGLINE:
{tagline}

GENERATED SUMMARY:
{summary}

Score on these dimensions. Be strict. Return ONLY valid JSON, no other text.

{{
  "faithfulness": <1-5>,
  "faithfulness_reason": "<one sentence>",

  "coverage": <1-5>,
  "coverage_reason": "<one sentence — are major themes from source represented?>",

  "no_fabrication": <1-5>,
  "no_fabrication_reason": "<one sentence — does summary avoid specific claims not in source?>",

  "thin_source_handled": <1-5>,
  "thin_source_reason": "<one sentence — if source is sparse, did summary stay short rather than pad?>",

  "tagline_accuracy": <1-5>,
  "tagline_reason": "<one sentence — is tagline factually grounded in the source text?>"
}}

Scoring guide:
5 = excellent, fully meets the criterion
4 = good, minor issues
3 = acceptable, noticeable issues  
2 = poor, significant problems
1 = fails criterion entirely
"""

def call_llm_judge(client , row: pd.Series) -> dict:
    # Truncate source text to keep costs low — first 3000 chars is enough for judge
    source_truncated = row[COL_SOURCE] #[:3000]

    prompt = JUDGE_PROMPT.format(
        source_text = source_truncated,
        tagline     = row[COL_TAGLINE],
        summary     = row[COL_SUMMARY],
    )

    try:

        body = json.dumps({
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 512,
            "anthropic_version": "bedrock-2023-05-31"
        })

        response = client.invoke_model(
            modelId='anthropic.claude-3-haiku-20240307-v1:0',
            body=body
        )
    
        response_body = json.loads(response['body'].read())
        text = response_body['content'][0]['text']

        # Strip markdown fences if present
        text = re.sub(r"^```json|^```|```$", "", text, flags=re.MULTILINE).strip()
        scores = json.loads(text)
        scores["llm_judge_error"] = ""
        return scores

    except json.JSONDecodeError as e:
        return {"llm_judge_error": f"JSON parse error: {e}", "raw_response": text}
    except Exception as e:
        return {"llm_judge_error": str(e)}

In [23]:
def stratified_sample(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """
    Sample across small/medium/large CBSAs.
    Uses city_count if available, else source text length as proxy.
    """
    df = df.copy()

    size_col = df[COL_SOURCE].str.len()

    df["_size_bucket"] = pd.qcut(size_col, q=3, labels=["small", "medium", "large"])

    per_bucket = n // 3
    sampled = (
        df.groupby("_size_bucket", group_keys=False)
          .apply(lambda x: x.sample(min(len(x), per_bucket), random_state=seed))
    )

    # Top up to n if rounding left us short
    if len(sampled) < n:
        remaining = df[~df.index.isin(sampled.index)]
        extra = remaining.sample(min(n - len(sampled), len(remaining)), random_state=seed)
        sampled = pd.concat([sampled, extra])

    return sampled.drop(columns=["_size_bucket"]).reset_index(drop=True)

In [ ]:
# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    # Load data
    print(f"Loading {INPUT_FILE}...")
    df = pd.read_csv(INPUT_FILE)

    print(f"Total CBSAs: {len(df)}")

    # Validate required columns
    for col in [COL_CBSA, COL_SOURCE, COL_SUMMARY, COL_TAGLINE]:
        assert col in df.columns, f"Missing column: {col}. Check CONFIG section."

    # Drop rows with missing values in key columns
    df = df.dropna(subset=[COL_SOURCE, COL_SUMMARY, COL_TAGLINE])
    print(f"CBSAs after dropping nulls: {len(df)}")

    # Stratified sample
    sample_df = stratified_sample(df, SAMPLE_SIZE, RANDOM_SEED)
    print(f"Sampled {len(sample_df)} CBSAs for evaluation (stratified by size)")

    # Init boto3 client
    client = boto3.client(
        service_name='bedrock-runtime',
        region_name='us-east-1'
    )


    results = []

    for i, row in sample_df.iterrows():
        print(f"[{i+1}/{len(sample_df)}] Evaluating: {row[COL_CBSA]}")
        result = {COL_CBSA: row[COL_CBSA]}
        
        # LLM judge
        llm_scores = call_llm_judge(client, row)
        result.update(llm_scores)

        results.append(result)
        time.sleep(SLEEP_BETWEEN)

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nResults saved to {OUTPUT_CSV}")

    # overalls scores
    llm_dims = ["faithfulness", "coverage", "no_fabrication", 
            "thin_source_handled", "tagline_accuracy"]

    results_df[llm_dims] = results_df[llm_dims].apply(pd.to_numeric, errors='coerce')
    results_df["overall_score"] = results_df[llm_dims].mean(axis=1)

    print(results_df[["cbsa_name", "overall_score"] + llm_dims].sort_values("overall_score"))
    print(f"\nMean overall score: {results_df['overall_score'].mean():.2f}/5")

   
main()

Loading ../data/processed/cbsa_wiki_wikivoyage_summaries_df.csv...
Total CBSAs: 935
CBSAs after dropping nulls: 932
Sampled 100 CBSAs for evaluation (stratified by size)
[1/100] Evaluating: Urbana, OH
[2/100] Evaluating: Altus, OK
[3/100] Evaluating: Cody, WY
[4/100] Evaluating: Cordele, GA
[5/100] Evaluating: Bay City, TX
[6/100] Evaluating: Corsicana, TX
[7/100] Evaluating: Fergus Falls, MN
[8/100] Evaluating: Mineral Wells, TX
[9/100] Evaluating: Russellville, AL
[10/100] Evaluating: Canton, IL
[11/100] Evaluating: Douglas, GA
[12/100] Evaluating: Logansport, IN
[13/100] Evaluating: Washington Court House, OH
[14/100] Evaluating: Yuba City, CA
[15/100] Evaluating: Troy, AL
[16/100] Evaluating: Uvalde, TX
[17/100] Evaluating: Albertville, AL
[18/100] Evaluating: Laurinburg, NC
[19/100] Evaluating: Ludington, MI
[20/100] Evaluating: Magnolia, AR
[21/100] Evaluating: Deming, NM
[22/100] Evaluating: Gainesville, FL
[23/100] Evaluating: Hutchinson, MN
[24/100] Evaluating: Oak Harbor, WA


In [ ]:
df = pd.read_csv("../data/evaluation/summary_eval_results.csv")

judge_dims = ["faithfulness", "coverage", "no_fabrication", 
              "thin_source_handled", "tagline_accuracy"]
df[judge_dims] = df[judge_dims].apply(pd.to_numeric, errors="coerce")
df["overall_score"] = df[judge_dims].mean(axis=1)

# Overall
print("=== Overall Scores ===")
for dim in judge_dims:
    print(f"  {dim}: {df[dim].mean():.2f}/5  ({(df[dim]>=4).mean()*100:.0f}% scoring ≥4)")
print(f"  overall:  {df['overall_score'].mean():.2f}/5")


# Worst 5
print("\n=== Lowest Scoring Summaries ===")
cols = ["cbsa_name", "overall_score", "faithfulness_reason", 
        "coverage_reason", "no_fabrication_reason"]
print(df.nsmallest(5, "overall_score")[cols].to_string(index=False))

=== Overall Scores ===
  faithfulness: 4.46/5  (98% scoring ≥4)
  coverage: 4.32/5  (97% scoring ≥4)
  no_fabrication: 4.63/5  (100% scoring ≥4)
  thin_source_handled: 4.48/5  (100% scoring ≥4)
  tagline_accuracy: 4.54/5  (98% scoring ≥4)
  overall:  4.49/5

=== Lowest Scoring Summaries ===
       cbsa_name  overall_score                                                                                                                                   faithfulness_reason                                                                                                                                                                                                       coverage_reason                                                                                                                                                no_fabrication_reason
      Dublin, GA            2.2                                                    The generated summary describes a city in Georgia, while the sou

## 2. Evaluating Retrieval (Semantic Search)

In [26]:
test_cases = [
    {
        "query": "city with live music and arts scene",
        "expected": ["Austin-Round Rock-San Marcos, TX"]
    },
    {
        "query": "research triangle",
        "expected": ["Raleigh-Cary, NC", "Durham-Chapel Hill, NC"]
    },
    {
        "query": "technical hub",
        "expected": ["Seattle-Tacoma-Bellevue, WA"]
    },
    {
        "query": "silicon valley",
        "expected": ["San Francisco-Oakland-Fremont, CA", "San Jose-Sunnyvale-Santa Clara, CA"]
    },
    {
        "query": "automotive hub",
        "expected": ["Detroit-Warren-Dearborn, MI"]
    }
]

def evaluate_retrieval(test_cases, top_k=5):
    for test in test_cases:
        results = semantic_search.search(test["query"], top_k=top_k)
        returned_cbsas = [r["cbsa"] for r in results]

        hits = [cbsa for cbsa in test["expected"] if cbsa in returned_cbsas]
        precision = len(hits) / len(test["expected"])

        print(f"\nQuery: {test['query']}")
        print(f"Expected : {test['expected']}")
        print(f"Returned : {returned_cbsas}")
        print(f"Hits     : {hits}")
        print(f"Precision: {precision:.2f}")

evaluate_retrieval(test_cases)


Query: city with live music and arts scene
Expected : ['Austin-Round Rock-San Marcos, TX']
Returned : ['Austin-Round Rock-San Marcos, TX', 'Torrington, CT', 'Tullahoma-Manchester, TN', 'Rockford, IL', 'Kansas City, MO-KS']
Hits     : ['Austin-Round Rock-San Marcos, TX']
Precision: 1.00

Query: research triangle
Expected : ['Raleigh-Cary, NC', 'Durham-Chapel Hill, NC']
Returned : ['Arecibo, PR', 'Los Alamos, NM', 'Durham-Chapel Hill, NC', 'Raleigh-Cary, NC', 'Greensboro-High Point, NC']
Hits     : ['Raleigh-Cary, NC', 'Durham-Chapel Hill, NC']
Precision: 1.00

Query: technical hub
Expected : ['Seattle-Tacoma-Bellevue, WA']
Returned : ['Lubbock, TX', 'Seattle-Tacoma-Bellevue, WA', 'Marion-Herrin, IL', 'Effingham, IL', 'Portland-Vancouver-Hillsboro, OR-WA']
Hits     : ['Seattle-Tacoma-Bellevue, WA']
Precision: 1.00

Query: silicon valley
Expected : ['San Francisco-Oakland-Fremont, CA', 'San Jose-Sunnyvale-Santa Clara, CA']
Returned : ['San Jose-Sunnyvale-Santa Clara, CA', 'Portland-Vanco

In [27]:
test_cases = [
    {
        "query": "Coastal big city with hiking trails"
    },
    {
        "query": "Large city with good public transit"
    },
    {
        "query": "Walkable city with nightlife",
    },
    {
        "query": "Urban city with lots of parks",
    },
    {
        "query": "Mountain city with outdoor activities",
    },
    {
        "query": "good place for young professionals",
    }
]
for test in test_cases:
    results = semantic_search.search(test["query"], top_k=5)
    print(f"\nQuery: {test['query']}")
    for r in results:
        print(f"  {r['cbsa']}")


Query: Coastal big city with hiking trails
  Traverse City, MI
  Michigan City-La Porte, IN
  San Diego-Chula Vista-Carlsbad, CA
  Panama City-Panama City Beach, FL
  Crestview-Fort Walton Beach-Destin, FL

Query: Large city with good public transit
  Chicago-Naperville-Elgin, IL-IN
  San Jose-Sunnyvale-Santa Clara, CA
  Atlanta-Sandy Springs-Roswell, GA
  Detroit-Warren-Dearborn, MI
  Hagerstown-Martinsburg, MD-WV

Query: Walkable city with nightlife
  Traverse City, MI
  Michigan City-La Porte, IN
  Las Vegas-Henderson-North Las Vegas, NV
  Urban Honolulu, HI
  Adrian, MI

Query: Urban city with lots of parks
  Michigan City-La Porte, IN
  Denver-Aurora-Centennial, CO
  Dodge City, KS
  Reading, PA
  Rapid City, SD

Query: Mountain city with outdoor activities
  Mountain Home, AR
  Bozeman, MT
  Traverse City, MI
  Breckenridge, CO
  Brigham City, UT-ID

Query: good place for young professionals
  Durham-Chapel Hill, NC
  Youngstown-Warren, OH
  Greensboro-High Point, NC
  Little Ro

The system is generally performing well in identifying relevant cities and understanding user lifestyle preferences such as coastal living, mountains, transit access, walkability, and nightlife. However, it struggles to clearly distinguish between city scale and type, particularly in separating large urban metros from smaller towns or rural areas. 

## 3. Evaluate Explanation

In [ ]:

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── CONFIG ────────────────────────────────────────────────────────────────────

OUTPUT_CSV    = "../data/evaluation/eval_rag_explanation_results.csv"
TOP_N         = 5      # evaluate top 5 recommendations per user combo
SLEEP_BETWEEN = 0.5    # seconds between Bedrock calls

THEME_SCORE_KEYS = [
    "affordability_score", "safety_score", "job_growth_score",
    "education_score", "health_score", "walkability_score",
    "diversity_score", "urban_score", "weather_warmth_score",
    "weather_mildness_score"
]

# ── SIMULATED USER PREFERENCE COMBINATIONS ───────────────────────────────────
# 5 diverse user types covering different priority patterns
# Sliders are 0-5; 5 = highest priority

TEST_CASES = [
    {
        "label": "young_professional",
        "query": "tech jobs, walkable city, vibrant nightlife",
        "prefs": {
            "affordability_score": 3,
            "safety_score": 3,
            "job_growth_score": 5,
            "education_score": 3,
            "health_score": 2,
            "walkability_score": 5,
            "diversity_score": 4,
            "urban_score": 5,
            "weather_warmth_score": 3,
            "weather_mildness_score": 2,
        }
    },
    {
        "label": "retiree",
        "query": "warm weather, safe, affordable for retirement",
        "prefs": {
            "affordability_score": 4,
            "safety_score": 5,
            "job_growth_score": 1,
            "education_score": 2,
            "health_score": 5,
            "walkability_score": 3,
            "diversity_score": 2,
            "urban_score": 2,
            "weather_warmth_score": 5,
            "weather_mildness_score": 4,
        }
    },
    {
        "label": "family_with_kids",
        "query": "good schools, safe neighborhoods, affordable housing",
        "prefs": {
            "affordability_score": 4,
            "safety_score": 5,
            "job_growth_score": 3,
            "education_score": 5,
            "health_score": 4,
            "walkability_score": 3,
            "diversity_score": 3,
            "urban_score": 2,
            "weather_warmth_score": 3,
            "weather_mildness_score": 3,
        }
    },
    {
        "label": "remote_worker_outdoors",
        "query": "outdoor activities, mild weather, low cost of living",
        "prefs": {
            "affordability_score": 5,
            "safety_score": 3,
            "job_growth_score": 2,
            "education_score": 2,
            "health_score": 3,
            "walkability_score": 3,
            "diversity_score": 2,
            "urban_score": 1,
            "weather_warmth_score": 3,
            "weather_mildness_score": 5,
        }
    },
    {
        "label": "all_high",
        "query": "best overall city to live in",
        "prefs": {
            "affordability_score": 5,
            "safety_score": 5,
            "job_growth_score": 5,
            "education_score": 5,
            "health_score": 5,
            "walkability_score": 5,
            "diversity_score": 5,
            "urban_score": 5,
            "weather_warmth_score": 5,
            "weather_mildness_score": 5,
        }
    },
]


In [ ]:

# ── HELPERS ───────────────────────────────────────────────────────────────────

def extract_theme_scores(city: dict) -> dict:
    """Extract only the 10 theme score keys from a city dict."""
    return {k: city[k] for k in THEME_SCORE_KEYS if k in city}


def get_bedrock_client():
    """Create boto3 Bedrock client using environment variables."""
    return boto3.client(
        service_name="bedrock-runtime",
        region_name="us-east-1")


In [26]:

# ── LLM JUDGE ─────────────────────────────────────────────────────────────────

JUDGE_PROMPT = """You are evaluating a personalized city recommendation explanation 
generated by an AI relocation tool.

The tool recommended this city from a pool of 900+ US metros/micros based on the user's 
combined preferences. These are already good matches — evaluate whether the 
explanation correctly reflects that.

USER PREFERENCES (0-5 scale, 5 = highest priority):
{user_prefs}

USER'S OWN WORDS: "{user_query}"

CBSA THEME SCORES (0-5 scale):
{theme_scores}

CBSA SUMMARY (the only context the explanation should use):
{cbsa_summary}

GENERATED EXPLANATION:
{explanation}

Score on three dimensions. Be strict. Return ONLY valid JSON, no other text.

{{
    "score_alignment": <1-5>,
    "score_alignment_reason": "<one sentence: does explanation correctly reflect numeric scores? flags contradictions like calling a 3/10 affordability score 'affordable'>",

    "grounding": <1-5>,
    "grounding_reason": "<one sentence: does explanation stay within the summary? flags outside knowledge about the city not present in the summary>",

    "personalization": <1-5>,
    "personalization_reason": "<one sentence: does explanation connect to THIS user's specific high-priority preferences, or is it generic enough to apply to any user?>"
}}

Scoring guide:
5 = excellent
4 = good, minor issues
3 = acceptable, noticeable issues
2 = poor, significant problems
1 = fails entirely
"""


In [27]:

def call_judge(client, user_prefs, user_query, theme_scores, cbsa_summary, explanation):
    """Call LLM judge to score a single explanation."""
    
    prefs_str = "\n".join(
        f"  {k.replace('_score','').replace('_',' ').title()}: {v}/5"
        for k, v in user_prefs.items()
    )
    scores_str = "\n".join(
        f"  {k.replace('_score','').replace('_',' ').title()}: {v:.1f}/5"
        for k, v in theme_scores.items()
    )

    prompt = JUDGE_PROMPT.format(
        user_prefs   = prefs_str,
        user_query   = user_query,
        theme_scores = scores_str,
        cbsa_summary = cbsa_summary,   # truncate to control cost
        explanation  = explanation,
    )

    try:
        body = json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 512,
            "messages": [{"role": "user", "content": prompt}],
        })

        response = client.invoke_model(
            modelId="anthropic.claude-3-haiku-20240307-v1:0",
            body=body,
        )

        response_body = json.loads(response["body"].read())
        raw = response_body["content"][0]["text"].strip()

        # Strip markdown fences if present
        import re
        raw = re.sub(r"^```json|^```|```$", "", raw, flags=re.MULTILINE).strip()
        scores = json.loads(raw)
        scores["judge_error"] = ""
        return scores

    except json.JSONDecodeError as e:
        logger.error("JSON parse error from judge: %s", e)
        return {"judge_error": f"JSON parse error: {e}"}
    except Exception as e:
        logger.error("Judge call failed: %s", e)
        return {"judge_error": str(e)}


In [28]:
# ── MAIN ──────────────────────────────────────────────────────────────────────

def main(df):
    """
    Args:
        df: Your standardized_indicies_df dataframe
    """
    client = get_bedrock_client()
    results = []

    for combo in TEST_CASES:
        label    = combo["label"]
        query    = combo["query"]
        prefs    = combo["prefs"]

        logger.info("Running combo: %s", label)

        # Get top N recommendations using your existing recommender
        results_df = recommend_cities(
            df          = df,
            user_inputs = prefs,
            user_query  = query,
            top_n       = TOP_N,
        )

        cities = add_text_to_cbsa(results_df).to_dict(orient="records")

        for city in cities:
            cbsa_name    = city.get("cbsa_name", city.get("cbsa", "unknown"))
            theme_scores = extract_theme_scores(city)
            cbsa_summary = city.get("summary", "")

            logger.info("  Generating explanation for: %s", cbsa_name)

            # Generate explanation
            explanation = generate_explanation(
                client      = client,
                user_prefs  = prefs,
                user_query  = query,
                cbsa_name   = cbsa_name,
                theme_scores= theme_scores,
                cbsa_summary= cbsa_summary,
            )
            time.sleep(SLEEP_BETWEEN)

            if not explanation:
                logger.warning("  No explanation generated for %s", cbsa_name)
                continue

            # Judge the explanation
            logger.info("  Judging explanation for: %s", cbsa_name)
            scores = call_judge(
                client       = client,
                user_prefs   = prefs,
                user_query   = query,
                theme_scores = theme_scores,
                cbsa_summary = cbsa_summary,
                explanation  = explanation,
            )
            time.sleep(SLEEP_BETWEEN)

            result = {
                "user_type"   : label,
                "user_query"  : query,
                "cbsa_name"   : cbsa_name,
                "explanation" : explanation,
            }
            result.update({f"pref_{k}": v for k, v in prefs.items()})
            result.update(theme_scores)
            result.update(scores)
            results.append(result)

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)
    logger.info("Results saved to %s", OUTPUT_CSV)

In [ ]:
# Load your dataframe here
standardized_indicies_df = pd.read_csv("../data/final/Final_Enriched_Dataset.csv")
main(standardized_indicies_df)


INFO:__main__:Running combo: young_professional
Batches: 100%|██████████| 1/1 [00:00<00:00, 28.43it/s]
INFO:__main__:  Generating explanation for: Atlantic City-Hammonton, NJ Metro Area


{'41860': 0.47232985496520996, '25540': 0.45490431785583496, '41940': 0.4532134532928467, '38900': 0.4360019564628601, '33100': 0.4231574535369873, '20500': 0.41344547271728516, '46520': 0.4100320339202881, '38300': 0.3996514081954956, '12060': 0.3918970823287964, '29820': 0.39063262939453125, '28420': 0.38944900035858154, '19100': 0.3804817199707031, '16860': 0.37848639488220215, '17410': 0.3783038854598999, '15180': 0.3768906593322754, '42660': 0.37258899211883545, '24660': 0.3719598650932312, '47900': 0.37179362773895264, '45900': 0.3700065612792969, '16740': 0.36996304988861084, '16700': 0.3690069913864136, '17780': 0.36507654190063477, '17420': 0.3600965142250061, '37980': 0.357743501663208, '10580': 0.35570192337036133, '28140': 0.3555619716644287, '40220': 0.3543996810913086, '42100': 0.3529577851295471, '30780': 0.3528108596801758, '16980': 0.34971219301223755, '31080': 0.34950369596481323, '42540': 0.3485180735588074, '47260': 0.3446909189224243, '43780': 0.3441869020462036, '

INFO:__main__:  Judging explanation for: Atlantic City-Hammonton, NJ Metro Area
INFO:__main__:  Generating explanation for: Roanoke, VA Metro Area
INFO:__main__:  Judging explanation for: Roanoke, VA Metro Area
INFO:__main__:  Generating explanation for: Miami-Fort Lauderdale-West Palm Beach, FL Metro Area
INFO:__main__:  Judging explanation for: Miami-Fort Lauderdale-West Palm Beach, FL Metro Area
INFO:__main__:  Generating explanation for: Pittsburgh, PA Metro Area
INFO:__main__:  Judging explanation for: Pittsburgh, PA Metro Area
INFO:__main__:  Generating explanation for: Portland-Vancouver-Hillsboro, OR-WA Metro Area
INFO:__main__:  Judging explanation for: Portland-Vancouver-Hillsboro, OR-WA Metro Area
INFO:__main__:Running combo: retiree
Batches: 100%|██████████| 1/1 [00:00<00:00, 11.99it/s]
INFO:__main__:  Generating explanation for: Yuma, AZ Metro Area


{'26300': 0.37829434871673584, '10480': 0.3382072448730469, '40340': 0.31996989250183105, '16220': 0.3096553087234497, '49740': 0.30812740325927734, '29460': 0.30319344997406006, '26140': 0.29394102096557617, '42200': 0.293662965297699, '48680': 0.2910717725753784, '37580': 0.28954726457595825, '28420': 0.2881101369857788, '46520': 0.28076332807540894, '39460': 0.27984362840652466, '43220': 0.2780213952064514, '17740': 0.27668821811676025, '42540': 0.2753458023071289, '33260': 0.2753407955169678, '21220': 0.27433347702026367, '21820': 0.27427423000335693, '25940': 0.2693561315536499, '34880': 0.2691265344619751, '22380': 0.2688896656036377, '39660': 0.26603955030441284, '41940': 0.26502156257629395, '38900': 0.264878511428833, '30820': 0.26430612802505493, '45880': 0.263297975063324, '15980': 0.263014018535614, '32940': 0.26254427433013916, '34940': 0.2621426582336426, '27780': 0.2619612216949463, '30020': 0.2618042826652527, '44220': 0.2599419355392456, '14580': 0.2592170238494873, '1

INFO:__main__:  Judging explanation for: Yuma, AZ Metro Area
INFO:__main__:  Generating explanation for: Lakeland-Winter Haven, FL Metro Area
INFO:__main__:  Judging explanation for: Lakeland-Winter Haven, FL Metro Area
INFO:__main__:  Generating explanation for: Casper, WY Metro Area
INFO:__main__:  Judging explanation for: Casper, WY Metro Area
INFO:__main__:  Generating explanation for: Woodward, OK Micro Area
INFO:__main__:  Judging explanation for: Woodward, OK Micro Area
INFO:__main__:  Generating explanation for: Hot Springs, AR Metro Area
INFO:__main__:  Judging explanation for: Hot Springs, AR Metro Area
INFO:__main__:Running combo: family_with_kids
Batches: 100%|██████████| 1/1 [00:00<00:00, 19.39it/s]
INFO:__main__:  Generating explanation for: Harrisonburg, VA Metro Area


...

{'43220': 0.3980063199996948, '45880': 0.39607328176498413, '43320': 0.3872939944267273, '10620': 0.3785139322280884, '47980': 0.37198716402053833, '41620': 0.36754554510116577, '28900': 0.3655126094818115, '26820': 0.36097240447998047, '16220': 0.35930097103118896, '22260': 0.35564547777175903, '33140': 0.35526537895202637, '42540': 0.35059452056884766, '42200': 0.3466673493385315, '33540': 0.34589362144470215, '10480': 0.34315216541290283, '48460': 0.33586132526397705, '14260': 0.3346879482269287, '45900': 0.33410006761550903, '41940': 0.33406174182891846, '47700': 0.33380812406539917, '14720': 0.3322264552116394, '34940': 0.33087241649627686, '28420': 0.33080780506134033, '19100': 0.32974404096603394, '30780': 0.32646405696868896, '39900': 0.3260246515274048, '40180': 0.3253822326660156, '48140': 0.32372748851776123, '17410': 0.32288336753845215, '47900': 0.3207278251647949, '47260': 0.3201303482055664, '43420': 0.3192266821861267, '21380': 0.31762927770614624, '34740': 0.3153983354

INFO:__main__:  Judging explanation for: Albemarle, NC Micro Area
INFO:__main__:  Generating explanation for: Emporia, KS Micro Area
INFO:__main__:  Judging explanation for: Emporia, KS Micro Area
INFO:__main__:  Generating explanation for: Shelton, WA Micro Area
INFO:__main__:  Judging explanation for: Shelton, WA Micro Area
INFO:__main__:  Generating explanation for: Sierra Vista-Douglas, AZ Metro Area
INFO:__main__:  Judging explanation for: Sierra Vista-Douglas, AZ Metro Area
INFO:__main__:  Generating explanation for: Klamath Falls, OR Micro Area
INFO:__main__:  Judging explanation for: Klamath Falls, OR Micro Area
INFO:__main__:Running combo: all_high
Batches: 100%|██████████| 1/1 [00:00<00:00, 18.98it/s]
INFO:__main__:  Generating explanation for: Cincinnati, OH-KY-IN Metro Area


{'41860': 0.46613073348999023, '37980': 0.4611223340034485, '29820': 0.44502681493759155, '46520': 0.4390721321105957, '17140': 0.43608880043029785, '19820': 0.4315781593322754, '47700': 0.4270421266555786, '28420': 0.4266732335090637, '28140': 0.4182262420654297, '19100': 0.417618989944458, '16980': 0.41639530658721924, '26420': 0.41308069229125977, '46540': 0.4095573425292969, '34980': 0.40805280208587646, '12060': 0.40510380268096924, '38300': 0.4016076326370239, '42100': 0.4009784460067749, '47900': 0.3982311487197876, '41660': 0.3942676782608032, '25540': 0.3928762674331665, '41700': 0.39227235317230225, '25180': 0.38924169540405273, '41940': 0.3887898921966553, '40900': 0.3875237703323364, '45780': 0.3869413137435913, '29620': 0.386785626411438, '23060': 0.38624465465545654, '10760': 0.38605165481567383, '27780': 0.38592827320098877, '18020': 0.38518792390823364, '33140': 0.3849322199821472, '33340': 0.3843952417373657, '40220': 0.3831964135169983, '26620': 0.38219505548477173, '

INFO:__main__:  Judging explanation for: Cincinnati, OH-KY-IN Metro Area
INFO:__main__:  Generating explanation for: Roanoke, VA Metro Area
INFO:__main__:  Judging explanation for: Roanoke, VA Metro Area
INFO:__main__:  Generating explanation for: Nashville-Davidson--Murfreesboro--Franklin, TN Metro Area
INFO:__main__:  Judging explanation for: Nashville-Davidson--Murfreesboro--Franklin, TN Metro Area
INFO:__main__:  Generating explanation for: Atlanta-Sandy Springs-Roswell, GA Metro Area
INFO:__main__:  Judging explanation for: Atlanta-Sandy Springs-Roswell, GA Metro Area
INFO:__main__:  Generating explanation for: Birmingham, AL Metro Area
INFO:__main__:  Judging explanation for: Birmingham, AL Metro Area
INFO:__main__:Results saved to ../data/evaluation/eval_rag_explanation_results.csv


Import your dataframe and call main(df) to run evaluation.


In [30]:

df = pd.read_csv("eval_rag_explanation_results.csv")

judge_dims = ["score_alignment", "grounding", "personalization"]
df[judge_dims] = df[judge_dims].apply(pd.to_numeric, errors="coerce")
df["overall_score"] = df[judge_dims].mean(axis=1)

# Overall
print("=== Overall Scores ===")
for dim in judge_dims:
    print(f"  {dim}: {df[dim].mean():.2f}/5  ({(df[dim]>=4).mean()*100:.0f}% scoring ≥4)")
print(f"  overall:  {df['overall_score'].mean():.2f}/5")

# By user type
print("\n=== By User Type ===")
print(df.groupby("user_type")["overall_score"].mean().round(2).to_string())

# Worst 5
print("\n=== Lowest Scoring Explanations ===")
cols = ["user_type", "cbsa_name", "overall_score", 
        "score_alignment_reason", "grounding_reason", "personalization_reason"]
print(df.nsmallest(5, "overall_score")[cols].to_string(index=False))

=== Overall Scores ===
  score_alignment: 3.52/5  (56% scoring ≥4)
  grounding: 4.24/5  (92% scoring ≥4)
  personalization: 4.12/5  (96% scoring ≥4)
  overall:  3.96/5

=== By User Type ===
user_type
all_high                  4.00
family_with_kids          4.20
remote_worker_outdoors    3.93
retiree                   3.73
young_professional        3.93

=== Lowest Scoring Explanations ===
         user_type                              cbsa_name  overall_score                                                                                                                                                     score_alignment_reason                                                                                                                                              grounding_reason                                                                                                                                                                    personalization_reason
           retiree  